# Agent 3: 연결 취약도 프로파일과 국소 교환

AI 연구 보조 산출물(2026-11-22). 실제 조원의 개인 기여로 표시하지 않는다.

연구 질문: 동일한 탐욕 초기해에 최대 세 번의 간선 교환을 추가하면 공식 최악 공격 점수가 개선되는가?
공개 합성 문제은행의 생성 시드 0,1에 대한 **개발·예비 비교**이다. 10개 그래프에 예산 두 개씩을 적용했으므로 20행을 독립 그래프 20개로 해석하지 않는다. 미사용 시드 확인실험은 별도이다.

이 노트북은 저장된 웹 입력으로 두 독립 소스를 다시 실행하고 출력·예산을 검증한 뒤 실제 웹 CSV를 분석한다. 웹에 새 제출을 만들지 않는다. 공식 점수와 한도 준수의 증거는 job 3 및 job 6의 CSV이며 노트북의 로컬 실행시간은 공식 측정치가 아니다.

데이터: 프로젝트 합성 문제은행과 공개 웹 https://algorithm.586729.xyz (2026-11-22 다운로드). 입력은 inputs/ 에 포함되고 생성·평가 계약은 저장소 specs/001-network-protection-study/spec.md를 따른다.

담당: Codex agent 3 (AI). 사람 검토 예정. 파일·프로토콜과 고정 소스 해시 확인.

In [1]:
from pathlib import Path
import csv, hashlib, io, json, runpy, subprocess, sys
root = Path.cwd()
if not (root / "protocol.json").exists():
    root = root / "results/agent-study/2026-11-22/agent-3"
protocol = json.loads((root / "protocol.json").read_text())
runs = json.loads((root / "web_runs.json").read_text())
for job in runs["jobs"]:
    source = root / (job["variant"] + ".py")
    assert hashlib.sha256(source.read_bytes()).hexdigest() == job["source_sha256"]
print("고정 사례:", len(protocol["cases"]), "웹 job:", [j["job_id"] for j in runs["jobs"]])

고정 사례: 20 웹 job: [3, 6]


담당: Codex agent 3 (AI). 입력 파싱·독립 프로그램 실행·출력과 예산 검증. 매 실행은 별도 프로세스이며 최적 참조를 전달하지 않는다.

In [2]:
checks = []
for variant in protocol["variants"]:
    with (root / (variant + ".csv")).open(encoding="utf-8-sig") as handle:
        official = {row["scenario"]: row for row in csv.DictReader(handle)}
    for case in protocol["cases"]:
        data = (root / "inputs" / (case + ".txt")).read_bytes()
        lines = data.decode().splitlines()
        n, m, d, a = map(int, lines[0].split())
        budget = int(lines[1].split()[0])
        assert all(len(line.split()) == 2 for line in lines[2:2+n])
        edges = {int(row.split()[0]): int(row.split()[4]) for row in lines[2+n:2+n+m]}
        proc = subprocess.run([sys.executable, str(root / (variant + ".py"))], input=data,
                              capture_output=True, check=True, timeout=10)
        count, *ids = map(int, proc.stdout.split())
        assert count == len(ids) and ids == sorted(set(ids))
        assert sum(edges[e] for e in ids) <= budget
        assert ids == list(map(int, official[case]["chosen_edges"].split()))
        checks.append((variant, case, count))
print("웹 제출과 동일 출력·예산 검증:", len(checks), "실행")

웹 제출과 동일 출력·예산 검증: 40 실행


담당: Codex agent 3 (AI). 실패 포함 전 사례 집계. 공식 점수는 연결 쌍 수 우선이며 두 점수를 임의로 가중 합하지 않는다.

In [3]:
analysis = runpy.run_path(str(root / "analyze.py"))["analyze"](root)
print(json.dumps(analysis["summary"], indent=2, ensure_ascii=False))
print("쌍별 비교:", analysis["paired_counts"])
print("변화 사례:")
for row in analysis["comparisons"]:
    if row["outcome"] != "tied":
        print(row)

{
  "greedy": {
    "total": 20,
    "completed": 20,
    "failures": [],
    "optimal_score_count": 1,
    "median_seconds": 0.07873602000472602,
    "max_seconds": 0.11825383099494502,
    "max_peak_mb": 6.11
  },
  "local_search": {
    "total": 20,
    "completed": 20,
    "failures": [],
    "optimal_score_count": 5,
    "median_seconds": 0.0833056244882755,
    "max_seconds": 0.11198613297892734,
    "max_peak_mb": 6.1
  }
}
쌍별 비교: {'improved': 5, 'tied': 14, 'worsened': 1, 'incomplete': 0}
변화 사례:
{'scenario': 'n8-seed0-p20', 'greedy_status': 'completed', 'local_search_status': 'completed', 'greedy_score': (15, 123596), 'local_search_score': (16, 483146), 'outcome': 'improved', 'connected_pairs_change': 1, 'delivery_units_change_when_connected_tied': None}
{'scenario': 'n8-seed1-p40', 'greedy_status': 'completed', 'local_search_status': 'completed', 'greedy_score': (28, 595960), 'local_search_score': (28, 484848), 'outcome': 'worsened', 'connected_pairs_change': 0, 'delivery_unit

## 해석과 한계

교환 개선은 20사례 중 5개 개선, 14개 동점, 1개 악화였다. 공식 최적 점수 일치는 1개에서 5개가 되었다. 두 소스 모두 20개 정상 완료했다. 대리 연결성 프로파일을 개선해도 전달량은 악화할 수 있으므로, 알고리즘의 공식 점수 단조 개선이나 최적성을 주장할 수 없다. 작은 결정적 표본의 결과이며 무작위 시드 확인실험과 불확실성 분석은 메인 에이전트가 별도로 통합한다. 웹 실행시간에는 프로세스 준비·입출력 시간이 포함되어 이 규모에서는 알고리즘 간 미세 시간 차이의 일반화가 어렵다.

관련 연구: Vaessens, Aarts, Lenstra (1998), *A local search template*, DOI 10.1016/S0305-0548(97)00093-2; Hansen, Mladenović (2001), *Variable neighborhood search: Principles and applications*, DOI 10.1016/S0377-2217(00)00100-4. 상세 서지·확인 URL·실제 열람 범위는 references.json. 본 구현은 고정 이웃의 제한적 개선이며 VNS 전체를 재현한 것이 아니다.

사람이 수행할 부분: 대리 지표와 공식 LP 차이를 설명하고, 두 논문 원문을 읽은 뒤 자기 말로 요약하며, 코드 수정과 재실행 내역을 개인 기여로 남긴다. AI 작성 코드·문장을 사람의 직접 작성 성과로 바꾸어 표시하지 않는다.